# __LANGCHAIN__

In [ ]:
pip install -r /home/aswanth.cp@knackforge.com/Public/Tutorial/udemy/langchain/requirements.txt -q

In [ ]:
pip install langchain-groq

In [ ]:
import os
from dotenv import load_dotenv, find_dotenv


load_dotenv(find_dotenv(), override=True)

In [ ]:

from langchain_groq import ChatGroq

llm = ChatGroq(
    model="llama-3.3-70b-versatile",
    api_key=os.getenv('GROQ_API_KEY')
)

response = llm.invoke("What is the speed of light?")
print(response.content)

In [ ]:

messages = [
    SystemMessage(content='You are a pysicist and respond only in German.'),
    HumanMessage(content='What is the speed of light?'),
]

output = llm.invoke(messages)

print(output.content)

## __CACHING IN LANGCHAIN__

In [ ]:
from langchain_core.globals import set_llm_cache
from langchain_community.cache import InMemoryCache


llm = ChatGroq(
    model="llama-3.3-70b-versatile",
    api_key=os.getenv('GROQ_API_KEY')
)

In [ ]:
%%time
set_llm_cache(InMemoryCache())
response = llm.invoke("What is the speed of light?")
print(response.content)

In [ ]:
%%time
response = llm.invoke("What is the speed of light?")
print(response.content)

## __LLM Streaming__

In [ ]:
for chunk in llm.stream("What is the speed of light?"):
    print(chunk.content, end="\n")

## __PROMPT Templates__

In [ ]:
from langchain_core.prompts import PromptTemplate


prompt_string = """

    You are a helpful assistant that translates English to German.
    Translate the following sentence to German: {sentence}

"""

prompt_template = PromptTemplate.from_template(prompt_string)

prompt = prompt_template.format(sentence="I am a Aswanth CP, from India.")

pro

In [ ]:

from langchain_groq import ChatGroq

llm = ChatGroq(
    model="llama-3.3-70b-versatile",
    api_key=os.getenv('GROQ_API_KEY')
)
output = llm.invoke(prompt)
print(output.content)

## __ChatPromptTemplates__


In [ ]:
from langchain_core.prompts import ChatPromptTemplate, HumanMessagePromptTemplate
from langchain_core.messages import SystemMessage

chat_template = ChatPromptTemplate.from_messages(
    [
        SystemMessage(content='You respond only in the json format.'),
        HumanMessagePromptTemplate.from_template("Top {n} countries in {area} by population."),
    ]
)

message = chat_template.format_messages(n=5, area="Asia")
print(message)

In [ ]:
message2 = chat_template.format_messages(n=5, area="South America")
output = llm.invoke(message2)
print(output.content)

## Simple Chain

In [ ]:
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_groq import ChatGroq
from langchain_core.globals import set_debug

output_parser = StrOutputParser()

llm = ChatGroq(
    model="llama-3.3-70b-versatile",
    api_key=os.getenv('GROQ_API_KEY')
)

template = '''

You are an experience virologist. 
Write a few sentencees about the following virus: {virus} in {language}.

'''

prompt_template = PromptTemplate.from_template(template)

chain = prompt_template | llm | output_parser

# set_debug(True)

result = chain.invoke(
{"virus": "Ebola", "language": "tamil"}
)
print(result)


In [ ]:
template = '''
 List the top {num_places} places to visit in the {city}. Use bullet points to list the places. Answer in {language}.
'''

prompt_template = PromptTemplate.from_template(template)
chain = prompt_template | llm | output_parser
result = chain.invoke(
{"num_places": 5, "city": "Chennai", "language": "English"}
)
print(result)

## Sequential Chains

In [ ]:
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_groq import ChatGroq

output_parser = StrOutputParser()

llm = ChatGroq(
    model="llama-3.3-70b-versatile",
    api_key=os.getenv('GROQ_API_KEY'),
    temperature=1.5,
 )

prompt_template = PromptTemplate.from_template(
    template="You are an experienced scientist and python programmer. Write a function that implements the concept of {concept}."
 )

chain1 = prompt_template | llm | output_parser

llm2 = ChatGroq(
    model="openai/gpt-oss-120b",
    api_key=os.getenv('GROQ_API_KEY'),
    temperature=0.5,
 )

prompt_template2 = PromptTemplate.from_template(
    template="Give the python function {function}, describe it as detailed as possible."
 )

chain2 = prompt_template2 | llm2 | output_parser

overall_chain = chain1 | (lambda output: {"function": output}) | chain2

output = overall_chain.invoke({"concept": "recursion"})
print(output)

In [ ]:
print(output)

# **LangChain Agents.**

In [ ]:
from langchain_experimental.tools import PythonREPLTool

python_tool = PythonREPLTool()

python_tool.run('print("Hello World")')


In [ ]:
from langchain_experimental.agents.agent_toolkits import create_python_agent
from langchain_experimental.tools import PythonREPLTool
from langchain_groq import ChatGroq


llm = ChatGroq(
    model="llama-3.3-70b-versatile",
    api_key=os.getenv('GROQ_API_KEY'),
    temperature=0.5,
)

agent_executor = create_python_agent(
    llm=llm,
    tool = PythonREPLTool(),
    verbose=True
)


agent_executor.run("What is the output of the following code: print(2+3*4)?")

In [ ]:
agent_executor.run("Calculate the square root of the factorial of 12 and display it with 4 decimal places.")

### LangChain Tools: DuckDuckGo and Wikipedia

In [ ]:
pip install -q duckduckgo-search ddgs

In [ ]:
from langchain_community.tools import DuckDuckGoSearchResults

search = DuckDuckGoSearchResults()

output = search.invoke('what is capital of India?')
print(output)

In [ ]:
search.name

In [ ]:
search.description

In [ ]:
from langchain_community.tools import DuckDuckGoSearchResults

search = DuckDuckGoSearchResults()

output = search.run('what is capital of India?')
print(output)

In [ ]:
from langchain_community.tools import DuckDuckGoSearchResults
from langchain_community.utilities import DuckDuckGoSearchAPIWrapper

wrapper = DuckDuckGoSearchAPIWrapper(
    region="us-en",
    max_results=5
)

search = DuckDuckGoSearchResults(
    api_wrapper=wrapper
)

output = search.run("New Delhi")
print(output)

In [ ]:
import re
print(output)
pattern = r"snippet: (.*?), title: (.*?), link: (.*?)(?=snippet:|$)"

matches = re.findall(pattern, output, re.DOTALL)
print(matches)
for snippet, title, link in matches:
    print(f"Title: {title}\nSnippet: {snippet}\nLink: {link}\n")
    print("-" * 50)

In [ ]:
pip install -q wikipedia

In [ ]:
from langchain_community.utilities import WikipediaAPIWrapper
from langchain_community.tools import WikipediaQueryRun

In [ ]:
api_wrapper = WikipediaAPIWrapper(top_k_results=1, doc_content_chars_max=500)

wiki = WikipediaQueryRun(api_wrapper=api_wrapper)
wiki.invoke({'query': 'virat kohli'})